# β stabilisation and mode type: GFTM (FILTER=0.5) and TGLF against GS2 (M1_A1_NE, R3, R4)

γ vs β at fixed ky, one figure per code against GS2, for each case. Codes: GFTM widths 0.4 / 0.6 / 0.7 / 0.8
(`scan_beta_f05`, FILTER=0.5) and TGLF default / V5 (`scan_beta_modes`). All scans use GS2's own (ky, β) grid.

**Line styles.** GS2 black (dominant mode only). Code: dominant mode (`argmax` of `growth_rate` over `mode`, never
index 0) solid, the other mode dashed (`NMODES=2`).

**Markers = mode type** from `general_analysis.mode_classification` (cutoffs `T_BAL`, `T_TEAR`, see that module):
▲ MTM (tearing parity), ■ KBM (ballooning parity), ● mixed. Markers are drawn only where γ > 0.
- **GS2** uses all four indicators (tearing parameter T, ω sign, χe/χi, De/χe) from each run's final time.
- **GFTM / TGLF** linear runs have no per-mode flux weights, so they are classified on **T and ω only**, with the
  same cutoffs: KBM = ω>0 and T<`T_BAL`; MTM = ω<0 and T>`T_TEAR`; otherwise mixed. A code marker is therefore a
  looser test than a GS2 one, and "KBM" there means only "ballooning parity, ion direction".
- T for GFTM / TGLF is the `tearing_parameter` stored per mode in each leaf's `pyroscan.nc`.

**β axis.** Code scans store pyrokinetics β (= GS2's number × a per-case factor f); the x axis here is **GS2's β
number**, matched by index. f is printed below and checked constant.

## Imports and settings

In [ ]:
from pathlib import Path
import os
import numpy as np
import xarray as xr
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pyrokinetics import Pyro
from pyrokinetics.diagnostics.field_line import FieldLine
from general_analysis import mode_classification as mc

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir()
)
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)

analysis_name = "beta_stabilisation_f05"
data_root = Path(os.environ["GK_DATA_ROOT"]).expanduser()
run_template = "Runs"

cases = ["M1_A1_NE", "R3", "R4"]
ky_targets = [0.1, 0.2, 0.3, 0.5]          # snapped to the nearest GS2 grid value, shown in the panel titles

# GS2 reference: data_root / code / run_template / project / case / scan_information
gs2_code, gs2_project, gs2_scan = "GS2", "scan_beta", "parameter_scan_GS2"
# Codes, one figure each: label -> (code, project, scan_information = leaf). Same three path parts for every case.
codes = {
    "GFTM w0.4": ("GFTM", "scan_beta_f05", "w0p4"),
    "GFTM w0.6": ("GFTM", "scan_beta_f05", "w0p6"),
    "GFTM w0.7": ("GFTM", "scan_beta_f05", "w0p7"),
    "GFTM w0.8": ("GFTM", "scan_beta_f05", "w0p8"),
    "TGLF default": ("TGLF", "scan_beta_modes", "default"),
    "TGLF V5": ("TGLF", "scan_beta_modes", "v5"),
}
deck = {"GFTM": "input.gftm", "TGLF": "input.tglf"}     # per-run input name, only for the marker spot check
gamma_floor = 0.0                                        # markers only where gamma > this
n_spot, seed = 3, 0                                      # spot-check cells per (case, code family)

## Load data

The scan `pyroscan.nc` of every leaf, opened lazily. GS2 and the codes must share the ky grid exactly; nothing is
interpolated.

In [ ]:
def scan(code, project, case, leaf):
    return xr.open_dataset(data_root / code / run_template / project / case / leaf / "pyroscan.nc")

gs2 = {c: scan(gs2_code, gs2_project, c, gs2_scan) for c in cases}
model = {(c, lab): scan(code, project, c, leaf) for c in cases for lab, (code, project, leaf) in codes.items()}

for c in cases:
    for lab in codes:
        ds = model[(c, lab)]
        assert np.array_equal(ds.ky.values, gs2[c].ky.values), (c, lab, "ky grid differs from GS2")
        ratio = ds.beta.values / gs2[c].beta.values
        assert np.allclose(ratio, ratio[0], rtol=1e-3), (c, lab, "beta is not GS2 beta x constant")
    print(c, "ky", gs2[c].ky.size, "beta", gs2[c].beta.size, "| pyro beta / GS2 beta f =", round(float(ratio[0]), 4),
          "| modes", {lab: int(model[(c, lab)].mode.size) for lab in codes})

## Calculate and select

Nearest GS2 ky to each target (actual values printed). GS2 type per plotted (ky, β) cell from the run's own
deck and output, through `mode_classification.indicators` + `classify`. Code type from stored `tearing_parameter` and
`mode_frequency` per mode with the same cutoffs (T and ω only, see header).

In [ ]:
ky_index = {c: [int(np.abs(gs2[c].ky.values - k).argmin()) for k in ky_targets] for c in cases}
for c in cases:
    print(c, "ky target -> GS2 grid:", {k: round(float(gs2[c].ky.values[i]), 4) for k, i in zip(ky_targets, ky_index[c])})

def gs2_type(case, ky, beta):
    "GS2 run directory is named from GS2's own ky and beta numbers (3 decimals)"
    d = data_root / gs2_code / run_template / gs2_project / case / gs2_scan / f"ky{ky:.3f}_beta{beta:.3f}"
    pyro = Pyro(gk_file=d / "gs2.in")
    pyro.load_gk_output(load_fields=True, load_fluxes=True)
    return mc.classify(mc.indicators(pyro))

gs2_label = {(c, i, j): gs2_type(c, float(gs2[c].ky[i]), float(gs2[c].beta[j]))
             for c in cases for i in ky_index[c] for j in range(gs2[c].beta.size)
             if gs2[c].growth_rate.values[i, j] > gamma_floor}

def code_type(T, omega):
    "same cutoffs as mode_classification.classify, on the indicators a GFTM/TGLF linear run has"
    w = mc.ION_DIRECTION * omega
    return np.where((w > 0) & (T < mc.T_BAL), "KBM", np.where((w < 0) & (T > mc.T_TEAR), "MTM", "mixed"))

code_label = {k: code_type(ds.tearing_parameter.values, ds.mode_frequency.values) for k, ds in model.items()}
dominant = {k: ds.growth_rate.fillna(-np.inf).argmax("mode").values for k, ds in model.items()}   # (ky, beta)
for c in cases:
    n = {t: sum(1 for k, v in gs2_label.items() if k[0] == c and v == t) for t in ("KBM", "MTM", "mixed")}
    print(c, "GS2 type counts at plotted ky (gamma>0):", n)

## Plot

One figure per (case, code): four ky panels, γ vs GS2-β. Markers: ▲ MTM, ■ KBM, ● mixed.

In [ ]:
marker = {"MTM": "^", "KBM": "s", "mixed": "o"}
key = [Line2D([], [], color="k", label="GS2 (dominant)"),
       Line2D([], [], color="C0", ls="-", label="code, dominant mode"),
       Line2D([], [], color="C0", ls="--", label="code, other mode"),
       *[Line2D([], [], color="grey", ls="", marker=m, label=t) for t, m in marker.items()]]
figs = {}
for c in cases:
    ref, beta = gs2[c], gs2[c].beta.values
    for lab in codes:
        ds, dom, typ = model[(c, lab)], dominant[(c, lab)], code_label[(c, lab)]
        fig, axes = plt.subplots(1, len(ky_targets), figsize=(4 * len(ky_targets), 3.6), sharey=True)
        for ax, i in zip(axes, ky_index[c]):
            ax.plot(beta, ref.growth_rate.values[i], color="k", marker="none")
            for j, g in enumerate(ref.growth_rate.values[i]):
                if (c, i, j) in gs2_label:
                    ax.plot(beta[j], g, color="k", marker=marker[gs2_label[(c, i, j)]], ls="")
            for m in range(ds.mode.size):
                g = ds.growth_rate.values[i, :, m]
                ax.plot(beta, g, color="C0", ls="-" if m == 0 else "--", marker="none")
            for j in range(beta.size):
                for m in range(ds.mode.size):
                    g = ds.growth_rate.values[i, j, m]
                    if g > gamma_floor:
                        ax.plot(beta[j], g, color="C0", marker=marker[typ[i, j, m]], ls="", mfc="C0" if m == dom[i, j] else "none")
            ax.set(xlabel=r"GS2 $\beta$", title=f"ky = {float(ref.ky[i]):.3f} (target {ky_targets[list(ky_index[c]).index(i)]})")
        axes[0].set_ylabel(r"$\gamma$ [$c_s/a$]")
        axes[-1].legend(handles=key, loc="best", fontsize=8)
        fig.suptitle(f"{c}: {lab} vs GS2. Dominant = solid line, filled marker; other mode dashed, open marker", y=1.04)
        figs[(c, lab)] = fig
        plt.close(fig)
print(len(figs), "figures")

## Save

In [ ]:
output_dir = ROOT / "Plots" / analysis_name
output_dir.mkdir(parents=True, exist_ok=True)
for (c, lab), fig in figs.items():
    fig.savefig(output_dir / f"{c}_{lab.replace(' ', '_').replace('.', 'p')}.png", dpi=150, bbox_inches="tight")
print("saved", len(figs), "to", output_dir)

## Marker spot check

For each (case, code family) three random plotted cells (seed above, gamma > 0). Code families: GFTM (4 widths pooled)
and TGLF (2 pooled). **Code cells**: T is recomputed from the run's own deck with pyro's
`FieldLine.compute_linear_tearing_parameter` and compared with the stored value and the marker type. **GS2 cells**: the
full indicator set is shown with the label it produced.

In [ ]:
rng = np.random.default_rng(seed)

def code_run_dir(case, lab, i, j):
    code, project, leaf = codes[lab]
    ds = model[(case, lab)]
    return (data_root / code / run_template / project / case / leaf /
            f"ky_{float(ds.ky[i]):.3f}" / f"beta_{float(ds.beta[j]):.3f}"), deck[code]

print("CODE cells: stored T vs recomputed T per mode, omega, label")
for c in cases:
    for family in ("GFTM", "TGLF"):
        labs = [l for l in codes if l.startswith(family)]
        cand = [(l, i, j, m) for l in labs for i in ky_index[c] for j in range(gs2[c].beta.size)
                for m in range(model[(c, l)].mode.size) if model[(c, l)].growth_rate.values[i, j, m] > gamma_floor]
        for k in rng.choice(len(cand), n_spot, replace=False):
            lab, i, j, m = cand[k]
            d, name = code_run_dir(c, lab, i, j)
            pyro = Pyro(gk_file=d / name)
            pyro.load_gk_output(load_fields=True, load_fluxes=False)
            T_new = float(np.asarray(FieldLine(pyro).compute_linear_tearing_parameter()).squeeze()[m])
            ds = model[(c, lab)]
            print(f"{c:9s} {lab:13s} ky={float(ds.ky[i]):.3f} beta_gs2={float(gs2[c].beta[j]):.2f} mode{m}"
                  f"  T stored {float(ds.tearing_parameter.values[i, j, m]):.4f} recomputed {T_new:.4f}"
                  f"  omega {float(ds.mode_frequency.values[i, j, m]):+.3f}  marker {code_label[(c, lab)][i, j, m]}")

print("GS2 cells: T, omega, chi_e/chi_i, De/chi_e -> label")
for c in cases:
    cand = [k for k in gs2_label if k[0] == c]
    for k in rng.choice(len(cand), n_spot, replace=False):
        _, i, j = cand[k]
        d = data_root / gs2_code / run_template / gs2_project / c / gs2_scan / f"ky{float(gs2[c].ky[i]):.3f}_beta{float(gs2[c].beta[j]):.3f}"
        pyro = Pyro(gk_file=d / "gs2.in")
        pyro.load_gk_output(load_fields=True, load_fluxes=True)
        ind = mc.indicators(pyro)
        print(f"{c:9s} GS2 ky={float(gs2[c].ky[i]):.3f} beta={float(gs2[c].beta[j]):.2f}  "
              + "  ".join(f"{n} {ind[n]:+.3f}" for n in ("T", "omega", "chi_ratio", "de_chi")) + f"  -> {mc.classify(ind)}")

## Interpretation

Left to the reader. Code-side markers use T and ω only (no flux weights), GS2 markers use all four indicators, so a
GS2 "mixed" next to a code "KBM" can be a difference in the test rather than in the physics.